# Skills: repo의 `.claude/skills` 자동으로 받아들이기

이미 Claude Code용 skill을 작성하는 팀은 repository의 `.claude/skills/<skill-name>/SKILL.md`에 이를 보관하며 설명하는 코드 옆에서 version을 관리합니다. 각각을 Skills API에 업로드하고 모든 agent의 version을 고정하는 것은 작업을 중복시키며 누군가 repo를 편집하는 순간 어긋납니다.

세션이 GitHub repository를 mount하면 harness는 세션 시작 시 repository 루트의 `.claude/skills/` 디렉터리를 scan하고 발견한 모든 skill을 agent의 system prompt에 주입합니다. skill의 이름, 설명, sandbox 내부 경로가 포함됩니다. 요청이 skill 설명과 일치하는 순간 model은 built-in `read` tool로 skill의 `SKILL.md`를 읽고, 함께 제공되는 script와 reference 파일을 포함하여 따릅니다. 업로드, agent의 `skills` 필드, version 관리가 필요 없습니다. checkout한 branch에 있는 skill을 받습니다.

이 notebook에서는 다음을 살펴봅니다.

- scanner가 기대하는 디스크 구조
- skill이 발견되도록 repository mount하기
- 발견을 확인하고 먼저 읽고 따르는 protocol 관찰하기
- 경계: 구조 규칙, skill 상한, 세션 도중 변경


## 1. client 설정하기

environment에 `GITHUB_TOKEN`이 필요합니다. 이 notebook이 mount하는 repository는 공개되어 있으므로 공개 repo 읽기 접근이 있는 fine-grained personal access token이면 충분합니다. repository skill은 Anthropic-hosted environment(`anthropic_cloud`)에서만 발견됩니다. self-hosted sandbox는 GitHub repository를 clone하지 않으므로 거기서는 이 기능을 다루지 않습니다.


In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
GH_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GH_TOKEN:
    raise SystemExit(
        "Set GITHUB_TOKEN in your environment (a fine-grained PAT with public-repo read)."
    )
client = anthropic.Anthropic()

## 2. scanner가 찾는 구조

skill은 repository 루트의 `.claude/skills/` 한 곳에서 발견됩니다. skill별로 디렉터리 하나이며 각 최상위에 `SKILL.md`가 있습니다. Claude Code와 같은 규칙입니다.

```
your-repo/
└── .claude/
    └── skills/
        ├── cookbook-audit/
        │   ├── SKILL.md          <- name + description frontmatter, 그 뒤 지시 사항
        │   ├── style_guide.md    <- 지시 사항이 가리키는 reference 파일
        │   └── validate_notebook.py  <- 지시 사항이 실행하는 script
        └── another-skill/
            └── SKILL.md
```

`SKILL.md`는 `name`과 `description`을 담은 YAML frontmatter로 시작합니다. description은 model이 파일을 열지 결정할 때 비교하는 내용이므로 trigger처럼 작성해야 합니다. skill이 무엇을 하고 언제 사용하는지를 나타내세요.

```markdown
---
name: cookbook-audit
description: Rubric을 기준으로 Anthropic Cookbook notebook을 audit합니다. Notebook review나 audit이 요청될 때마다 사용하세요.
---

# Cookbook Audit
...
```

공개 `anthropics/claude-cookbooks` repository는 `style_guide.md`의 평가 rubric과 `validate_notebook.py` script와 함께 바로 이 skill을 제공합니다. 이 repository를 mount하면 지시 사항, reference 파일, 실행 가능한 script를 갖춘 skill을 한 번에 얻습니다.


## 3. skill이 연결되지 않은 agent 생성하기

agent에는 skill을 참조하는 내용이 없습니다. 표준 toolset과 범용 prompt를 받습니다. skill은 agent 정의가 아니라 repository와 함께 옵니다.


In [ ]:
env = client.beta.environments.create(
    name="repo-skills-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

reviewer = client.beta.agents.create(
    name="cookbook_reviewer",
    description="Reviews notebooks in a mounted cookbook repository.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-use-skills-from-a-repo"},
    model={"id": MODEL},
    system="You review notebooks in the repository mounted under /workspace. "
    "When the repository provides a process for a task, follow it.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print(f"{reviewer.name}: {reviewer.id} v{reviewer.version}")

## 4. repository mount하기

`github_repository` resource는 세션 시작 시 repo를 container에 clone합니다. `url`은 일반 `https://github.com/{owner}/{repo}` 형태입니다(`.git` 접미사와 SSH 형태 없음). `checkout`은 branch나 commit을 고정하고 `mount_path`는 기본값 `/workspace/<repo-name>`을 덮어씁니다. skill 발견은 checkout한 ref를 따르므로 skill이 다른 branch에서는 다른 skill을 얻습니다.


In [ ]:
session = client.beta.sessions.create(
    agent=reviewer.id,
    environment_id=env.id,
    title="Cookbook audit",
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/anthropics/claude-cookbooks",
            "authorization_token": GH_TOKEN,
            "checkout": {"type": "branch", "name": "main"},
            # mount_path 기본값은 /workspace/claude-cookbooks입니다
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)

## 5. agent에 어떤 skill이 있는지 묻기

세션 시작 시 이미 발견했으므로 이 질문에는 파일 탐색 없이 답할 수 있습니다. agent는 system prompt에서 바로 `cookbook-audit`와 해당 `SKILL.md` 경로를 제시합니다.


In [ ]:
from utilities import stream_until_end_turn

client.beta.sessions.events.send(
    session.id,
    events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "What skills are available to you in this repo?"}],
        }
    ],
    betas=BETAS,
)
stream_until_end_turn(client, session.id)

## 6. 이름을 말하지 않고 skill 실행 유도하기

description의 목적은 model이 의도를 기준으로 routing하게 하는 것입니다. 이 요청은 "skill"이나 "cookbook-audit"를 언급하지 않습니다. skill이 만드는 결과를 요청합니다.

tool event를 살펴보세요. turn의 첫 `read`가 skill의 `SKILL.md`를 엽니다. agent는 자체 audit 과정을 만들어 내는 대신 skill의 지시를 따릅니다. `style_guide.md`를 읽고, `validate_notebook.py`를 실행하고, rubric에 따라 평가합니다.


In [ ]:
client.beta.sessions.events.send(
    session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Do a notebook audit of "
                    "managed_agents/CMA_iterate_fix_failing_tests.ipynb "
                    "against the cookbook style guide, and give me the score.",
                }
            ],
        }
    ],
    betas=BETAS,
)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            target = ev.input.get("file_path") or ev.input.get("command") or ""
            print(f"[{ev.name}] {str(target)[:100]}")
        elif ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

`read` 호출을 추려 protocol 자체를 확인하세요. 먼저 skill의 `SKILL.md`, 다음으로 제공되는 reference와 script, 그다음 작업에 관련된 repository 파일입니다.


In [ ]:
reads = [
    ev.input.get("file_path")
    for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS)
    if ev.type == "agent.tool_use" and ev.name == "read"
]
for path in reads:
    print(path)

## 7. scanner가 적용하는 규칙

**구조는 엄격합니다.** 세션 시작 시 repository 루트의 `.claude/skills/<name>/SKILL.md`만 안내됩니다. 하위 디렉터리 없는 `.claude/skills/SKILL.md`, 두 단계로 중첩된 `.claude/skills/a/b/SKILL.md`, `.claude` 상위 디렉터리 없는 `skills/<name>/SKILL.md`는 모두 무시됩니다.

**Frontmatter는 관대합니다.** frontmatter가 없는 `SKILL.md`도 디렉터리 이름을 이름으로, 빈 설명으로 안내됩니다. 긴 설명은 한 줄로 펼쳐지고 2,000자 이후는 잘립니다. CRLF 줄바꿈도 정상적으로 parsing됩니다.

**상한이 있습니다.** 처음 64개 skill 디렉터리가 안내됩니다. 이후 디렉터리는 오류 없이 제외되므로 큰 skill 모음이 있는 repository는 세션에 필요한 skill을 앞쪽에 적은 수로 유지해야 합니다.

**scan은 한 번 실행됩니다.** 발견은 세션 시작 시 이루어지며 해당 세션에 보존됩니다. 세션 도중 repository에 새 skill을 push해도 실행 중인 세션에는 영향을 주지 않습니다. 해당 repository로 생성하는 다음 세션이 이를 봅니다.

**중첩된 skill은 필요할 때 발견됩니다.** `packages/foo/.claude/skills/lint/SKILL.md`가 있고 루트 수준 skill이 없는 repository는 시작 시 아무것도 안내하지 않습니다. agent가 built-in `read` tool로 `packages/foo/` 아래 파일을 읽으면 harness가 중첩된 `.claude/skills/` 디렉터리를 알아차리고 남은 세션 동안 `lint`를 제공합니다. 같은 파일을 `bash`(`cat`)로 읽으면 유발되지 않습니다. `read` tool만 이를 유발합니다.


## 8. Skills API와 함께 사용하는 repository skill

두 방식은 한 세션에 공존하며 두 종류 모두 agent의 목록에 나타납니다.

skill이 코드와 함께 있어야 할 때 repository skill을 사용하세요. build와 test 규칙, review checklist, deploy runbook, source와 함께 pull request에서 변경하고 싶은 모든 것이 해당합니다. 사람이든 agent든 repo를 clone하면 같은 지시를 받습니다.

skill이 여러 repository에 걸치는 조직 전체 자산이거나, codebase와 독립적인 자체 versioning이 필요하거나, repository를 전혀 mount하지 않는 세션에서도 제공되어야 한다면 [Skills API](https://platform.claude.com/docs/en/managed-agents/skills)(agent의 `skills=[{"type": "custom", "skill_id": ...}]`)를 사용하세요.

둘 사이에 이름 충돌은 허용됩니다. repository의 `verify` skill과 조직의 `verify` skill이 서로 다른 경로로 모두 나타나며 model은 요청에 따라 선택합니다. mount한 두 repository가 같은 이름의 skill을 제공하면 서로 다른 mount 경로로 둘 다 안내됩니다.


## 9. 정리하기


In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")